In [35]:
# SoccerNet Dataset Preparation

#This notebook prepares the SoccerNet tracking dataset for the Computer
#Vision-Based Football Player Tracking and Movement Analysis project.

#The purpose of this stage is to obtain suitable football tracking sequences,
#inspect the dataset structure, select appropriate samples, and store the
#required data in persistent Google Drive storage for later player tracking,
#pitch calibration, and movement analysis#

In [36]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Football_CV_Project"
)

SOCCERNET_DRIVE = PROJECT_ROOT / "data/raw/soccernet"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"

SOCCERNET_DRIVE.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nSoccerNet permanent storage:")
print(SOCCERNET_DRIVE)

Project root:
/content/drive/MyDrive/Football_CV_Project

SoccerNet permanent storage:
/content/drive/MyDrive/Football_CV_Project/data/raw/soccernet


In [37]:
SOCCERNET_ZIP = Path(
    "/content/SoccerNet/tracking-2023/train.zip"
)

if SOCCERNET_ZIP.exists():

    size_gb = SOCCERNET_ZIP.stat().st_size / (1024 ** 3)

    print("SoccerNet training ZIP found!")
    print(f"Size: {size_gb:.2f} GB")

else:

    print("SoccerNet training ZIP is not available in this runtime.")

SoccerNet training ZIP found!
Size: 8.92 GB


In [38]:
!pip install -q SoccerNet

In [39]:
from SoccerNet.Downloader import SoccerNetDownloader

downloader = SoccerNetDownloader(
    LocalDirectory="/content/SoccerNet"
)

print("SoccerNet downloader ready.")

SoccerNet downloader ready.


In [40]:
downloader.downloadDataTask(
    task="tracking-2023",
    split=["train"]
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [41]:
SOCCERNET_ZIP = Path(
    "/content/SoccerNet/tracking-2023/train.zip"
)

print("ZIP exists:", SOCCERNET_ZIP.exists())

if SOCCERNET_ZIP.exists():
    size_gb = SOCCERNET_ZIP.stat().st_size / (1024 ** 3)
    print(f"ZIP size: {size_gb:.2f} GB")

ZIP exists: True
ZIP size: 8.92 GB


In [42]:
import zipfile

with zipfile.ZipFile(SOCCERNET_ZIP, "r") as z:
    files = z.namelist()

print("Total files:", len(files))
print("\nFirst 100 entries:\n")

for f in files[:100]:
    print(f)

Total files: 43207

First 100 entries:

train/
train/SNMOT-060/
train/SNMOT-060/gt/
train/SNMOT-060/gt/gt.txt
train/SNMOT-060/det/
train/SNMOT-060/det/det.txt
train/SNMOT-060/seqinfo.ini
train/SNMOT-060/gameinfo.ini
train/SNMOT-060/img1/
train/SNMOT-060/img1/000001.jpg
train/SNMOT-060/img1/000002.jpg
train/SNMOT-060/img1/000003.jpg
train/SNMOT-060/img1/000004.jpg
train/SNMOT-060/img1/000005.jpg
train/SNMOT-060/img1/000006.jpg
train/SNMOT-060/img1/000007.jpg
train/SNMOT-060/img1/000008.jpg
train/SNMOT-060/img1/000009.jpg
train/SNMOT-060/img1/000010.jpg
train/SNMOT-060/img1/000011.jpg
train/SNMOT-060/img1/000012.jpg
train/SNMOT-060/img1/000013.jpg
train/SNMOT-060/img1/000014.jpg
train/SNMOT-060/img1/000015.jpg
train/SNMOT-060/img1/000016.jpg
train/SNMOT-060/img1/000017.jpg
train/SNMOT-060/img1/000018.jpg
train/SNMOT-060/img1/000019.jpg
train/SNMOT-060/img1/000020.jpg
train/SNMOT-060/img1/000021.jpg
train/SNMOT-060/img1/000022.jpg
train/SNMOT-060/img1/000023.jpg
train/SNMOT-060/img1/00002

In [43]:
from collections import Counter
import os

extensions = Counter(
    os.path.splitext(f)[1].lower()
    for f in files
    if not f.endswith("/")
)

print("File types inside SoccerNet:")
print(extensions)

File types inside SoccerNet:
Counter({'.jpg': 42750, '.txt': 114, '.ini': 114})


In [44]:
#img1/ contains the actual consecutive football frames.
#gt/gt.txt contains ground-truth tracking annotations, which will become particularly useful when we evaluate our tracking system
#later.
#det/det.txt contains provided detections, while the .ini files contain sequence/game information.

In [45]:
# Find all SoccerNet training sequences

sequences = sorted({
    f.split("/")[1]
    for f in files
    if f.startswith("train/")
    and len(f.split("/")) > 1
    and f.split("/")[1].startswith("SNMOT-")
})

print("Number of training sequences:", len(sequences))
print("\nSequences:")

for seq in sequences:
    print(seq)

Number of training sequences: 57

Sequences:
SNMOT-060
SNMOT-061
SNMOT-062
SNMOT-063
SNMOT-064
SNMOT-065
SNMOT-066
SNMOT-067
SNMOT-068
SNMOT-069
SNMOT-070
SNMOT-071
SNMOT-072
SNMOT-073
SNMOT-074
SNMOT-075
SNMOT-076
SNMOT-077
SNMOT-097
SNMOT-098
SNMOT-099
SNMOT-100
SNMOT-101
SNMOT-102
SNMOT-103
SNMOT-104
SNMOT-105
SNMOT-106
SNMOT-107
SNMOT-108
SNMOT-109
SNMOT-110
SNMOT-111
SNMOT-112
SNMOT-113
SNMOT-114
SNMOT-115
SNMOT-151
SNMOT-152
SNMOT-153
SNMOT-154
SNMOT-155
SNMOT-156
SNMOT-157
SNMOT-158
SNMOT-159
SNMOT-160
SNMOT-161
SNMOT-162
SNMOT-163
SNMOT-164
SNMOT-165
SNMOT-166
SNMOT-167
SNMOT-168
SNMOT-169
SNMOT-170


In [46]:
from collections import Counter

frame_counts = Counter()

for f in files:
    parts = f.split("/")

    if (
        len(parts) >= 4
        and parts[0] == "train"
        and parts[2] == "img1"
        and f.lower().endswith(".jpg")
    ):
        frame_counts[parts[1]] += 1

print("Sequence frame counts:\n")

for seq in sequences:
    print(f"{seq}: {frame_counts[seq]} frames")

Sequence frame counts:

SNMOT-060: 750 frames
SNMOT-061: 750 frames
SNMOT-062: 750 frames
SNMOT-063: 750 frames
SNMOT-064: 750 frames
SNMOT-065: 750 frames
SNMOT-066: 750 frames
SNMOT-067: 750 frames
SNMOT-068: 750 frames
SNMOT-069: 750 frames
SNMOT-070: 750 frames
SNMOT-071: 750 frames
SNMOT-072: 750 frames
SNMOT-073: 750 frames
SNMOT-074: 750 frames
SNMOT-075: 750 frames
SNMOT-076: 750 frames
SNMOT-077: 750 frames
SNMOT-097: 750 frames
SNMOT-098: 750 frames
SNMOT-099: 750 frames
SNMOT-100: 750 frames
SNMOT-101: 750 frames
SNMOT-102: 750 frames
SNMOT-103: 750 frames
SNMOT-104: 750 frames
SNMOT-105: 750 frames
SNMOT-106: 750 frames
SNMOT-107: 750 frames
SNMOT-108: 750 frames
SNMOT-109: 750 frames
SNMOT-110: 750 frames
SNMOT-111: 750 frames
SNMOT-112: 750 frames
SNMOT-113: 750 frames
SNMOT-114: 750 frames
SNMOT-115: 750 frames
SNMOT-151: 750 frames
SNMOT-152: 750 frames
SNMOT-153: 750 frames
SNMOT-154: 750 frames
SNMOT-155: 750 frames
SNMOT-156: 750 frames
SNMOT-157: 750 frames
SNMOT-15

## Visual Inspection and Sequence Selection

A representative frame from each SoccerNet training sequence is displayed
to identify suitable sequences for player tracking and pitch-based movement
analysis. Sequences with a wide camera view, visible pitch markings, and
multiple players are preferred.

In [47]:
import zipfile
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import math

# Number of columns in the contact sheet
cols = 4
rows = math.ceil(len(sequences) / cols)

plt.figure(figsize=(20, rows * 3.5))

with zipfile.ZipFile(SOCCERNET_ZIP, "r") as z:

    for i, seq in enumerate(sequences):

        # Find all image frames belonging to this sequence
        seq_frames = [
            f for f in files
            if f.startswith(f"train/{seq}/img1/")
            and f.lower().endswith(".jpg")
        ]

        if len(seq_frames) == 0:
            continue

        # Sort frames into correct chronological order
        seq_frames.sort()

        # Choose the middle frame
        middle_frame = seq_frames[len(seq_frames) // 2]

        # Read that single image directly from the ZIP
        image_data = z.read(middle_frame)
        img = Image.open(BytesIO(image_data))

        # Add it to the contact sheet
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(seq)
        plt.axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## Selected Development Sequences

Three SoccerNet tracking sequences were selected for initial system
development based on their camera viewpoint, pitch visibility, player
coverage, and visibility of pitch markings.

Selected sequences:

- SNMOT-060
- SNMOT-066
- SNMOT-069

These sequences will be stored permanently in Google Drive and used for
player tracking and subsequent pitch-based movement analysis.

In [48]:
import zipfile
from pathlib import Path

selected_sequences = [
    "SNMOT-060",
    "SNMOT-066",
    "SNMOT-069"
]

SOCCERNET_SELECTED = (
    PROJECT_ROOT /
    "data/raw/soccernet"
)

with zipfile.ZipFile(SOCCERNET_ZIP, "r") as z:

    for seq in selected_sequences:

        print(f"\nExtracting {seq}...")

        prefix = f"train/{seq}/"

        sequence_files = [
            f for f in z.namelist()
            if f.startswith(prefix)
        ]

        for file in sequence_files:

            # Remove "train/" from the path
            relative_path = Path(file).relative_to("train")

            destination = (
                SOCCERNET_SELECTED /
                relative_path
            )

            if file.endswith("/"):
                destination.mkdir(
                    parents=True,
                    exist_ok=True
                )
                continue

            destination.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            with z.open(file) as source, \
                 open(destination, "wb") as target:

                target.write(source.read())

        print(
            f"{seq} saved - "
            f"{len(sequence_files)} files"
        )

print("\nSelected SoccerNet sequences saved successfully!")


Extracting SNMOT-060...
SNMOT-060 saved - 758 files

Extracting SNMOT-066...
SNMOT-066 saved - 758 files

Extracting SNMOT-069...
SNMOT-069 saved - 758 files

Selected SoccerNet sequences saved successfully!


In [49]:
for seq in selected_sequences:

    seq_path = SOCCERNET_SELECTED / seq
    image_path = seq_path / "img1"

    images = list(image_path.glob("*.jpg"))

    print(f"\n{seq}")
    print("Frames:", len(images))
    print("GT available:", (seq_path / "gt/gt.txt").exists())
    print("Detections available:", (seq_path / "det/det.txt").exists())
    print("Sequence info:", (seq_path / "seqinfo.ini").exists())


SNMOT-060
Frames: 750
GT available: True
Detections available: True
Sequence info: True

SNMOT-066
Frames: 750
GT available: True
Detections available: True
Sequence info: True

SNMOT-069
Frames: 750
GT available: True
Detections available: True
Sequence info: True


## Sequence Metadata Inspection

The metadata of the selected SoccerNet sequences is inspected to determine
their frame rate, resolution, sequence length, and image format before
computer vision processing.

In [50]:
import configparser

for seq in selected_sequences:

    ini_path = (
        SOCCERNET_SELECTED /
        seq /
        "seqinfo.ini"
    )

    config = configparser.ConfigParser()
    config.read(ini_path)

    print(f"\n===== {seq} =====")

    for key, value in config["Sequence"].items():
        print(f"{key}: {value}")


===== SNMOT-060 =====
name: SNMOT-060
imdir: img1
framerate: 25
seqlength: 750
imwidth: 1920
imheight: 1080
imext: .jpg

===== SNMOT-066 =====
name: SNMOT-066
imdir: img1
framerate: 25
seqlength: 750
imwidth: 1920
imheight: 1080
imext: .jpg

===== SNMOT-069 =====
name: SNMOT-069
imdir: img1
framerate: 25
seqlength: 750
imwidth: 1920
imheight: 1080
imext: .jpg


In [51]:
import pandas as pd

sequence_info = []

for seq in selected_sequences:

    ini_path = (
        SOCCERNET_SELECTED /
        seq /
        "seqinfo.ini"
    )

    config = configparser.ConfigParser()
    config.read(ini_path)

    info = config["Sequence"]

    fps = int(info["frameRate"])
    frames = int(info["seqLength"])
    width = int(info["imWidth"])
    height = int(info["imHeight"])

    duration = frames / fps

    sequence_info.append({
        "Sequence": seq,
        "Frames": frames,
        "FPS": fps,
        "Duration (sec)": round(duration, 2),
        "Resolution": f"{width} × {height}"
    })

summary_df = pd.DataFrame(sequence_info)

summary_df

,Sequence,Frames,FPS,Duration (sec),Resolution
0,SNMOT-060,750,25,30.0,1920 × 1080
1,SNMOT-066,750,25,30.0,1920 × 1080
2,SNMOT-069,750,25,30.0,1920 × 1080


In [52]:
metadata_path = (
    PROJECT_ROOT /
    "docs/results/soccernet_selected_sequences.csv"
)

# Create the folder if it doesn't already exist
metadata_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Save the CSV
summary_df.to_csv(
    metadata_path,
    index=False
)

print("Metadata summary saved:")
print(metadata_path)

Metadata summary saved:
/content/drive/MyDrive/Football_CV_Project/docs/results/soccernet_selected_sequences.csv


In [53]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Football_CV_Project"
)

print("=== FINAL NOTEBOOK 2 CHECK ===\n")

# Check selected SoccerNet sequences
selected_sequences = [
    "SNMOT-060",
    "SNMOT-066",
    "SNMOT-069"
]

for seq in selected_sequences:

    seq_path = (
        PROJECT_ROOT /
        "data/raw/soccernet" /
        seq
    )

    frames = list((seq_path / "img1").glob("*.jpg"))

    print(f"{seq}:")
    print(f"  Folder exists: {seq_path.exists()}")
    print(f"  Frames saved: {len(frames)}")
    print(f"  Ground truth: {(seq_path / 'gt/gt.txt').exists()}")
    print(f"  Sequence info: {(seq_path / 'seqinfo.ini').exists()}")
    print()

# Check metadata CSV
csv_path = (
    PROJECT_ROOT /
    "docs/results/soccernet_selected_sequences.csv"
)

print("Metadata CSV saved:", csv_path.exists())

=== FINAL NOTEBOOK 2 CHECK ===

SNMOT-060:
  Folder exists: True
  Frames saved: 750
  Ground truth: True
  Sequence info: True

SNMOT-066:
  Folder exists: True
  Frames saved: 750
  Ground truth: True
  Sequence info: True

SNMOT-069:
  Folder exists: True
  Frames saved: 750
  Ground truth: True
  Sequence info: True

Metadata CSV saved: True


In [54]:
#

In [55]:
print("PROJECT_ROOT =", PROJECT_ROOT)
print("SOCCERNET_SELECTED =", SOCCERNET_SELECTED)

print("\nProject root exists:", PROJECT_ROOT.exists())
print("SoccerNet folder exists:", SOCCERNET_SELECTED.exists())

PROJECT_ROOT = /content/drive/MyDrive/Football_CV_Project
SOCCERNET_SELECTED = /content/drive/MyDrive/Football_CV_Project/data/raw/soccernet

Project root exists: True
SoccerNet folder exists: True


In [56]:
!find "/content/drive/MyDrive/Football_CV_Project" -maxdepth 4 -type d | head -100

/content/drive/MyDrive/Football_CV_Project
/content/drive/MyDrive/Football_CV_Project/data
/content/drive/MyDrive/Football_CV_Project/data/raw
/content/drive/MyDrive/Football_CV_Project/data/raw/soccernet
/content/drive/MyDrive/Football_CV_Project/data/raw/soccernet/SNMOT-066
/content/drive/MyDrive/Football_CV_Project/data/raw/soccernet/SNMOT-069
/content/drive/MyDrive/Football_CV_Project/data/raw/soccernet/SNMOT-060
/content/drive/MyDrive/Football_CV_Project/data/processed
/content/drive/MyDrive/Football_CV_Project/docs
/content/drive/MyDrive/Football_CV_Project/docs/results


In [57]:
from pathlib import Path

for seq in ["SNMOT-060", "SNMOT-066", "SNMOT-069"]:

    path = Path(
        f"/content/drive/MyDrive/Football_CV_Project/data/raw/soccernet/{seq}"
    )

    print("\n", seq)
    print("Folder exists:", path.exists())

    if path.exists():
        frames = list((path / "img1").glob("*.jpg"))
        print("Frames:", len(frames))


 SNMOT-060
Folder exists: True
Frames: 750

 SNMOT-066
Folder exists: True
Frames: 750

 SNMOT-069
Folder exists: True
Frames: 750


In [58]:
docs_path = Path(
    "/content/drive/MyDrive/Football_CV_Project/docs"
)

print("Docs exists:", docs_path.exists())

if docs_path.exists():
    for p in docs_path.rglob("*"):
        print(p)

Docs exists: True
/content/drive/MyDrive/Football_CV_Project/docs/results
/content/drive/MyDrive/Football_CV_Project/docs/results/soccernet_selected_sequences.csv


In [59]:
!ls -lah "/content/drive/MyDrive/Football_CV_Project"
!ls -lah "/content/drive/MyDrive/Football_CV_Project/data/raw/soccernet"
!ls -lah "/content/drive/MyDrive/Football_CV_Project/docs/results"

total 20K
drwxr-xr-x 4 root root 4.0K Sep 23 08:02 .
drwxr-xr-x 3 root root 4.0K Sep 23 05:24 ..
drwxr-xr-x 4 root root 4.0K Sep 23 05:24 data
drwxr-xr-x 3 root root 4.0K Sep 23 06:23 docs
-rw-r--r-- 1 root root   87 Sep 23 08:02 DRIVE_TEST.txt
total 20K
drwxr-xr-x 5 root root 4.0K Sep 23 06:10 .
drwxr-xr-x 3 root root 4.0K Sep 23 05:24 ..
drwxr-xr-x 5 root root 4.0K Sep 23 06:10 SNMOT-060
drwxr-xr-x 5 root root 4.0K Sep 23 06:10 SNMOT-066
drwxr-xr-x 5 root root 4.0K Sep 23 06:10 SNMOT-069
total 12K
drwxr-xr-x 2 root root 4.0K Sep 23 06:23 .
drwxr-xr-x 3 root root 4.0K Sep 23 06:23 ..
-rw-r--r-- 1 root root  151 Sep 23 08:09 soccernet_selected_sequences.csv


In [60]:
test_file = PROJECT_ROOT / "DRIVE_TEST.txt"

with open(test_file, "w") as f:
    f.write("If you can see this file in Google Drive, Colab is writing to the correct Drive folder.")

print("Created:", test_file)
print("Exists:", test_file.exists())

Created: /content/drive/MyDrive/Football_CV_Project/DRIVE_TEST.txt
Exists: True


In [61]:
import os

print("Drive mounted:", os.path.ismount("/content/drive"))
print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))

print("\nContents of MyDrive:")
print(os.listdir("/content/drive/MyDrive")[:30])

Drive mounted: False
MyDrive exists: True

Contents of MyDrive:
['Football_CV_Project']


In [62]:
!df -h /content/drive

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   30G   78G  28% /


In [63]:
from google.colab import drive

drive.flush_and_unmount()
print("Drive unmounted.")

Drive not mounted, so nothing to flush and unmount.
Drive unmounted.


In [64]:
import os

os.rename(
    "/content/drive",
    "/content/local_drive_backup"
)

print("Temporary files protected.")

Temporary files protected.


In [65]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [66]:
import os

print("Drive mounted:", os.path.ismount("/content/drive"))

Drive mounted: True


In [67]:
from pathlib import Path

LOCAL_PROJECT = Path(
    "/content/local_drive_backup/MyDrive/Football_CV_Project"
)

REAL_PROJECT = Path(
    "/content/drive/MyDrive/Football_CV_Project"
)

print("Temporary project exists:", LOCAL_PROJECT.exists())
print("Real Drive project path:", REAL_PROJECT)

Temporary project exists: True
Real Drive project path: /content/drive/MyDrive/Football_CV_Project


In [68]:
import shutil

REAL_PROJECT.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copytree(
    LOCAL_PROJECT,
    REAL_PROJECT,
    dirs_exist_ok=True
)

print("Project successfully copied to Google Drive!")

Project successfully copied to Google Drive!


In [69]:
selected_sequences = [
    "SNMOT-060",
    "SNMOT-066",
    "SNMOT-069"
]

for seq in selected_sequences:

    seq_path = (
        REAL_PROJECT /
        "data/raw/soccernet" /
        seq
    )

    frames = list(
        (seq_path / "img1").glob("*.jpg")
    )

    print(f"\n{seq}")
    print("Folder exists:", seq_path.exists())
    print("Frames saved:", len(frames))
    print(
        "Ground truth:",
        (seq_path / "gt/gt.txt").exists()
    )
    print(
        "Sequence info:",
        (seq_path / "seqinfo.ini").exists()
    )


SNMOT-060
Folder exists: True
Frames saved: 750
Ground truth: True
Sequence info: True

SNMOT-066
Folder exists: True
Frames saved: 750
Ground truth: True
Sequence info: True

SNMOT-069
Folder exists: True
Frames saved: 750
Ground truth: True
Sequence info: True


In [70]:
csv_path = (
    REAL_PROJECT /
    "docs/results/soccernet_selected_sequences.csv"
)

baseline_video = (
    REAL_PROJECT /
    "data/raw/custom_clips/ronaldo_bicycle_kick_baseline.mp4"
)

print("Metadata CSV:", csv_path.exists())
print("Baseline Ronaldo video:", baseline_video.exists())

Metadata CSV: True
Baseline Ronaldo video: True
